# LLM Clinical Report Generation
**Genomic-RawSeq-Analyzer — Semester 2**

Final stage of the FASTQ → Deep Learning → **LLM Clinical Report** pipeline.

Uses **Meta-Llama-3-8B-Instruct** via HuggingFace `transformers` with 4-bit quantization
(fits in Colab T4 GPU ~15 GB). Generates structured clinical-style summaries for the
top 5 patients by cancer probability.

**Prerequisites:**
1. Run `OcclusionAnalysis.ipynb` first — needs `results/occlusion/motifs.json`
2. Accept the LLaMA-3 license on HuggingFace: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
3. Add your HF token to Colab Secrets: `Secrets → HF_TOKEN`

**Outputs saved to Google Drive:**
- `results/reports/<patient_id>_report.txt`  for each patient

In [9]:
# ── Setup ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json
import numpy as np
import pandas as pd
from datetime import datetime

!pip install -q transformers accelerate bitsandbytes tensorflow
from tensorflow.keras.models import load_model

# Load HuggingFace token from Colab Secrets
# Go to: Runtime → Secrets (🔑 icon) → Add new secret → Name: HF_TOKEN
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF token loaded.' if os.environ.get('HF_TOKEN') else 'ERROR: HF_TOKEN not set in Secrets!')

BASE        = '/content/drive/MyDrive/DNA_Anomaly_Detection'
BATCH_DIR   = f'{BASE}/BreastCancer_Data_Parts'
MODEL_PATH  = f'{BASE}/ML Models/BreastCancer_CNN_Model.keras'
MOTIFS_PATH = f'{BASE}/results/occlusion/motifs.json'
OUTPUT_DIR  = f'{BASE}/results/reports'
N_PATIENTS  = 5
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Inline: load_all_batches ──────────────────────────────────────────
def load_all_batches(batch_dir):
    files = sorted(glob.glob(os.path.join(batch_dir, 'batch_*.npz')))
    if not files:
        raise FileNotFoundError(f'No batch_*.npz files in {batch_dir}')
    X_parts, y_parts, id_parts = [], [], []
    for f in files:
        print(f'Loading {os.path.basename(f)}...')
        with np.load(f, allow_pickle=True) as d:
            X_parts.append(d['X'])
            y_parts.append(d['y'])
            if 'run_ids' in d:
                id_parts.append(d['run_ids'])
            else:
                n = len(d['X'])
                name = os.path.basename(f).replace('.npz', '')
                id_parts.append(np.array([f'{name}_read_{i}' for i in range(n)]))
    X = np.concatenate(X_parts)
    y = np.concatenate(y_parts)
    run_ids = np.concatenate(id_parts)
    print(f'Total: X={X.shape}  Tumor={int(y.sum()):,}  Normal={int((y==0).sum()):,}')
    return X, y, run_ids

print('Setup OK. Batch files:', sorted(glob.glob(f'{BATCH_DIR}/batch_*.npz')))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF token loaded.
Setup OK. Batch files: ['/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_1.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_2.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_3.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_4.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_5.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_6.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_7.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_8.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_9.npz']


## Step 1 — Build Patient List from CNN Predictions

In [10]:
print('Loading model and data...')
cnn = load_model(MODEL_PATH)
X, y, run_ids = load_all_batches(BATCH_DIR)

print('Running inference...')
probs = cnn.predict(X, batch_size=2048, verbose=1).flatten()

df = pd.DataFrame({'run_id': run_ids, 'prob': probs, 'label': y})
pat = df.groupby('run_id').agg(
    cancer_prob=('prob', 'mean'),
    patient_label=('label', lambda x: int(x.mode()[0])),
    n_reads=('prob', 'count'),
).reset_index().sort_values('cancer_prob', ascending=False)

# Load population motifs from OcclusionAnalysis
motifs = []
if os.path.exists(MOTIFS_PATH):
    with open(MOTIFS_PATH) as f:
        motifs = json.load(f).get('top_motifs', [])
    print(f'Loaded {len(motifs)} motifs from {MOTIFS_PATH}')
else:
    print(f'WARNING: {MOTIFS_PATH} not found. Run OcclusionAnalysis.ipynb first!')
    print('Proceeding with placeholder motifs for demonstration.')
    motifs = [
        {'position': 45, 'kmer': 'GCATC', 'score': 0.72, 'cosmic_hits': []},
        {'position': 61, 'kmer': 'TTCAG', 'score': 0.65, 'cosmic_hits': []},
        {'position': 12, 'kmer': 'CGATT', 'score': 0.58, 'cosmic_hits': []},
    ]

def risk_label(p):
    if p >= 0.65: return 'HIGH RISK'
    if p >= 0.55: return 'MODERATE RISK'
    return 'LOW RISK'

patients = []
for _, row in pat.head(N_PATIENTS).iterrows():
    ctype = 'Tumor (WXS)' if row['patient_label'] == 1 else 'Normal (WXS)'
    patients.append({
        'patient_id':  row['run_id'],
        'cancer_prob': round(float(row['cancer_prob']), 4),
        'n_reads':     int(row['n_reads']),
        'cancer_type': ctype,
        'top_motifs':  motifs,
    })

print(f'\nTop {N_PATIENTS} patients selected:')
for p in patients:
    print(f"  {p['patient_id']}  P(cancer)={p['cancer_prob']:.4f}  [{risk_label(p['cancer_prob'])}]  {p['cancer_type']}")

Loading model and data...
Loading batch_1.npz...
Loading batch_2.npz...
Loading batch_3.npz...
Loading batch_4.npz...
Loading batch_5.npz...
Loading batch_6.npz...
Loading batch_7.npz...
Loading batch_8.npz...
Loading batch_9.npz...
Total: X=(1800000, 80)  Tumor=1,100,000  Normal=700,000
Running inference...
879/879 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
Loaded 5 motifs from /content/drive/MyDrive/DNA_Anomaly_Detection/results/occlusion/motifs.json

Top 5 patients selected:
  batch_3_read_126425  P(cancer)=0.9826  [HIGH RISK]  Tumor (WXS)
  batch_1_read_101153  P(cancer)=0.9748  [HIGH RISK]  Tumor (WXS)
  batch_8_read_100271  P(cancer)=0.9734  [HIGH RISK]  Tumor (WXS)
  batch_9_read_100141  P(cancer)=0.9726  [HIGH RISK]  Tumor (WXS)
  batch_9_read_235  P(cancer)=0.9722  [HIGH RISK]  Tumor (WXS)


## Step 2 — Load LLAMA-3-8B-Instruct (4-bit quantized)
This cell downloads ~5 GB on first run. Subsequent runs use the Colab cache.

In [11]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_MODEL      = 'meta-llama/Meta-Llama-3-8B-Instruct'
LLM_CACHE_PATH = f'{BASE}/ML Models/llama3-8b-instruct-4bit'

# Free CNN from GPU before loading LLM
try:
    del cnn
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f'GPU after CNN cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

if os.path.exists(LLM_CACHE_PATH):
    print('Loading from Drive cache (fast)...')
    load_path = LLM_CACHE_PATH
else:
    print(f'First run — downloading {LLM_MODEL} from HuggingFace (~5 GB)...')
    load_path = LLM_MODEL

tokenizer = AutoTokenizer.from_pretrained(load_path, token=os.environ['HF_TOKEN'])
tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    load_path,
    quantization_config=bnb_config,
    device_map='auto',
    token=os.environ['HF_TOKEN'],
)
print(f'Model loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Save to Drive on first download so future sessions skip the HF download
if load_path == LLM_MODEL:
    print('Saving model to Drive for future sessions...')
    tokenizer.save_pretrained(LLM_CACHE_PATH)
    llm.save_pretrained(LLM_CACHE_PATH)
    print(f'Saved to {LLM_CACHE_PATH}. Future runs will load from Drive.')

GPU after CNN cleanup: 5.71 GB
First run — downloading meta-llama/Meta-Llama-3-8B-Instruct from HuggingFace (~5 GB)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded. GPU memory: 9.3 GB
Saving model to Drive for future sessions...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/DNA_Anomaly_Detection/ML Models/llama3-8b-instruct-4bit. Future runs will load from Drive.


## Step 3 — Prompt Template & Report Generation

In [15]:
SYSTEM_PROMPT = (
    'You are a clinical genomics assistant helping researchers interpret '
    'alignment-free whole-exome sequencing results. Your reports are '
    'informational summaries for research purposes only — not clinical diagnoses. '
    'Write clearly and concisely. Use a structured format with short sections.'
)

def build_prompt(patient):
    pid   = patient['patient_id']
    prob  = patient['cancer_prob']
    label = risk_label(prob)
    reads = patient.get('n_reads', 0)
    ctype = patient.get('cancer_type', 'WXS')

    motif_lines = []
    for m in patient.get('top_motifs', []):
        pos  = m['position']
        kmer = m['kmer']
        k    = len(kmer)
        sc   = m['score']
        hits = m.get('cosmic_hits', [])
        if hits:
            sigs = ', '.join(f"{h['signature']} ({h['description']})" for h in hits)
            motif_lines.append(f'  - Positions {pos}-{pos+k-1}: {kmer} (importance={sc:.3f}) → {sigs}')
        else:
            motif_lines.append(f'  - Positions {pos}-{pos+k-1}: {kmer} (importance={sc:.3f})')

    motif_block = '\n'.join(motif_lines) if motif_lines else '  - No high-importance motifs detected.'

    return (
        f'PATIENT GENOMIC ANALYSIS REPORT\n'
        f'================================\n'
        f'Patient ID        : {pid}\n'
        f'Sequencing Type   : {ctype}\n'
        f'Reads Analysed    : {reads:,} reads\n'
        f'Cancer Probability: {prob:.4f}  →  {label}\n'
        f'\nTop Occlusion-Sensitivity Motifs (k=5):\n{motif_block}\n'
        f'\n---\n'
        f'Based on the above alignment-free deep learning analysis, write\n'
        f'a structured clinical summary with these sections:\n'
        f'1. PATIENT SUMMARY\n'
        f'2. MODEL EVIDENCE (explain crowd-voting aggregation)\n'
        f'3. MOTIF ANALYSIS (COSMIC associations if any)\n'
        f'4. LIMITATIONS\n'
        f'5. RECOMMENDATION\n'
        f'Keep each section to 2-4 sentences. Do not invent data.'
    )


def generate_report(patient, max_new_tokens=700):
    prompt = build_prompt(patient)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    # apply_chat_template returns a BatchEncoding — extract the tensor with .input_ids
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to(llm.device)

    # If it came back as a BatchEncoding, unwrap it
    if hasattr(input_ids, 'input_ids'):
        input_ids = input_ids.input_ids

    with torch.no_grad():
        output_ids = llm.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def format_and_save(patient, llm_text, output_dir):
    pid   = patient['patient_id']
    prob  = patient['cancer_prob']
    ts    = datetime.now().strftime('%Y-%m-%d %H:%M')
    sep   = '=' * 70
    report = (
        f'{sep}\n'
        f'GENOMIC-RAWSEQ-ANALYZER — CLINICAL SUMMARY REPORT\n'
        f'{sep}\n'
        f'Patient ID   : {pid}\n'
        f'Cancer Prob. : {prob:.4f}  [{risk_label(prob)}]\n'
        f'Generated    : {ts}\n'
        f'DISCLAIMER   : Research use only. Not a clinical diagnosis.\n'
        f'{sep}\n\n'
        + llm_text.strip() +
        f'\n\n{sep}\nEND OF REPORT — {pid}\n{sep}\n'
    )
    path = os.path.join(output_dir, f'{pid}_report.txt')
    with open(path, 'w') as f:
        f.write(report)
    return path

print('Prompt template and generator ready.')

Prompt template and generator ready.


## Step 4 — Generate Reports for All Patients

In [16]:
import traceback

results = []
for i, patient in enumerate(patients, 1):
    pid = patient['patient_id']
    print(f'\n[{i}/{len(patients)}] Generating report for {pid}...')
    try:
        text = generate_report(patient)
        path = format_and_save(patient, text, OUTPUT_DIR)
        results.append((pid, path, 'OK'))
        print(f'  Saved: {path}')
    except Exception as e:
        tb = traceback.format_exc()
        results.append((pid, None, tb))
        print(f'  ERROR:\n{tb}')

print(f'\n{"="*50}')
print('REPORT GENERATION COMPLETE')
print(f'{"="*50}')
for pid, path, status in results:
    print(f'  {pid}  →  {path if path else "FAILED"}')

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



[1/5] Generating report for batch_3_read_126425...


[transformers] Both `max_new_tokens` (=700) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=700) and `max_length`(=4

  Saved: /content/drive/MyDrive/DNA_Anomaly_Detection/results/reports/batch_3_read_126425_report.txt

[2/5] Generating report for batch_1_read_101153...


[transformers] Both `max_new_tokens` (=700) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Saved: /content/drive/MyDrive/DNA_Anomaly_Detection/results/reports/batch_1_read_101153_report.txt

[3/5] Generating report for batch_8_read_100271...


[transformers] Both `max_new_tokens` (=700) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Saved: /content/drive/MyDrive/DNA_Anomaly_Detection/results/reports/batch_8_read_100271_report.txt

[4/5] Generating report for batch_9_read_100141...


[transformers] Both `max_new_tokens` (=700) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Saved: /content/drive/MyDrive/DNA_Anomaly_Detection/results/reports/batch_9_read_100141_report.txt

[5/5] Generating report for batch_9_read_235...
  Saved: /content/drive/MyDrive/DNA_Anomaly_Detection/results/reports/batch_9_read_235_report.txt

REPORT GENERATION COMPLETE
  batch_3_read_126425  →  /content/drive/MyDrive/DNA_Anomaly_Detection/results/reports/batch_3_read_126425_report.txt
  batch_1_read_101153  →  /content/drive/MyDrive/DNA_Anomaly_Detection/results/reports/batch_1_read_101153_report.txt
  batch_8_read_100271  →  /content/drive/MyDrive/DNA_Anomaly_Detection/results/reports/batch_8_read_100271_report.txt
  batch_9_read_100141  →  /content/drive/MyDrive/DNA_Anomaly_Detection/results/reports/batch_9_read_100141_report.txt
  batch_9_read_235  →  /content/drive/MyDrive/DNA_Anomaly_Detection/results/reports/batch_9_read_235_report.txt


## Step 5 — Display Sample Report

In [17]:
# Print the first generated report
first_pid, first_path, _ = results[0]
if first_path and os.path.exists(first_path):
    with open(first_path) as f:
        print(f.read())
else:
    print('No report to display.')

GENOMIC-RAWSEQ-ANALYZER — CLINICAL SUMMARY REPORT
Patient ID   : batch_3_read_126425
Cancer Prob. : 0.9826  [HIGH RISK]
Generated    : 2026-06-07 00:57
DISCLAIMER   : Research use only. Not a clinical diagnosis.

PATIENT GENOMIC ANALYSIS REPORT

**PATIENT SUMMARY**

The patient's whole-exome sequencing data suggests a high risk of cancer (probability: 0.9826). The analysis was performed using a deep learning model on 1 reads.

**MODEL EVIDENCE**

The model employed a crowd-voting aggregation strategy, where multiple neural networks were trained on large datasets and their predictions were combined to produce a consensus output. This approach enhances the model's robustness and accuracy in identifying potential cancer-related motifs.

**MOTIF ANALYSIS**

The top occlusion-sensitivity motifs identified in the analysis are: AGGCT (positions 28-32), CATTC (positions 8-12), AGAAA (positions 6-10), ATGTA (positions 20-24), and TGGAA (positions 10-14). While these motifs do not have direct CO